# 02 — Track access charge calibration

Per-country TAC component parameters, written to `data/tac_components.csv`
(plus `tac_night_mode.csv`, `tac_peak_bands.csv`, `passage_charges.csv`).
Narrative reasoning lives in `TAC_MODEL.md`; this notebook carries the numbers
and their pointers, so a tariff revision touches one line here rather than a
paragraph of prose. The implemented calculation is
`backend/models/infrastructure/calc_tac.py`; its design decisions are in
`TAC_CALC_DESIGN.md`.

**Scope.** Minimum access package only. Traction energy is `03`, shunting and
parking `04`, terrain and buffer `05`. Station and stop charges are out of
scope entirely (the CH Haltezuschlag is a capacity element of the path price,
not a station charge — it belongs here).

Every European formula reduces to the same shape once the national decoration
is stripped:

```
TAC_leg = train_km x B(band) + gross_tonne_km x gamma
        + stops x per_stop + seat_km x seat_rate
        + revenue_share x leg_revenue + fixed_per_train_km x train_km
        + peak surcharge (CH: x2 on the day rate, AT: flat EUR/train-km)
```

`night_mode` records how time enters — only two mechanisms exist:
`time_band` picks the rate per country run from the clock (DE, BE, IT, PT —
pro-rata against the country's band in `NIGHT_BANDS`, never a midpoint
pick), `none` has no documented differentiation. Germany's SPFV rule is a
band *widening*, not a third mechanism: a train carrying night accommodation
is priced Nacht over its entire German run (`night_full_if_accommodation`).
CH deliberately sits in `none` — its 22:00–06:00 band belongs to electricity
pricing, not TAC.


In [ ]:
# STDLIB-ONLY cell. See the seed-export contract in calib/README.md.
import csv
from pathlib import Path


def _calib_dir() -> Path:
    """Anchor on the calib folder whatever the kernel's cwd happens to be."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "resolution.py").exists():
            return cand
    for sub in ("backend/models/infrastructure/calib", "models/infrastructure/calib"):
        if (cwd / sub).is_dir():
            return (cwd / sub).resolve()
    raise RuntimeError("cannot locate models/infrastructure/calib")


CALIB_DIR = _calib_dir()
DATA_DIR = CALIB_DIR / "tac" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def write_data(name: str, fieldnames: list[str], rows: list[dict]) -> None:
    """Write one committed observation table to calib/tac/data/."""
    with open(DATA_DIR / name, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"tac/data/{name}: {len(rows)} rows")


def read_data(name: str) -> list[dict]:
    with open(DATA_DIR / name, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

In [ ]:
# STDLIB-ONLY cell.
from dataclasses import dataclass, asdict
from typing import Optional

# How much evidence stands behind a value. Not a quality judgement — a
# well-argued ASSUMED and a mis-transcribed SOURCED are both possible; the
# status says which kind of thing the reader is looking at.
SOURCED = "sourced"  # named document, named locator
NOT_LEVIED = "not_levied"  # positively documented as zero — not absent data
DERIVED = "derived"  # arithmetic on other values, formula in the note
BENCHMARK = "benchmark"  # pan-European statistic standing in for a country
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
MISSING = "missing"  # nothing read yet — never an estimate
NO_RAILWAY = "no_railway"

USABLE = {SOURCED, NOT_LEVIED, DERIVED, BENCHMARK, ASSUMED}


@dataclass(frozen=True)
class SV:
    """
    One calibrated parameter with its audit trail.

    source_id points at sources_register.csv; locator is what makes it
    re-checkable a year later (section, table, sheet), because a network
    statement runs to hundreds of pages and its tariff tables move between
    editions. Values stay in native currency and price basis — conversion and
    escalation happen later and explicitly.
    """

    country_code: str
    parameter: str
    value: Optional[float]
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    currency: str = "EUR"
    basis_year: Optional[int] = None
    note: str = ""
    low: Optional[float] = None  # sensitivity band, mandatory when ASSUMED
    high: Optional[float] = None

    def __post_init__(self):
        if self.status in USABLE and self.value is None:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: {self.status} needs a value"
            )
        if self.status in (SOURCED, NOT_LEVIED) and not (
            self.source_id and self.locator
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: sourced needs source_id + locator"
            )
        if self.status == ASSUMED and (
            self.low is None or self.high is None or not self.note
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: assumed needs a band and a rationale"
            )
        if self.status == DERIVED and not self.note:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: derived must state its formula"
            )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "currency",
    "basis_year",
    "note",
    "low",
    "high",
]


def emit(name: str, values: list[SV]) -> None:
    write_data(name, SV_FIELDS, [asdict(v) for v in values])
    by_status: dict[str, int] = {}
    for v in values:
        by_status[v.status] = by_status.get(v.status, 0) + 1
    for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
        print(f"    {s:11} {n:4}")

## Reference night train

Charging formulas are train-specific, so comparing countries needs one fixed
consist — the same role the reference route plays in the composition
calibration.

In [ ]:
NT_REF = dict(
    label="NT-REF",
    gross_weight_t=600.0,
    length_m=300.0,
    places=500,
    max_speed_kmh=160,
    n_locos=1,
    electric=True,
    market_segment="non-PSO open access",
    time_band="night 22:00-06:00",
    line_category="conventional",
)

# Per-country night tariff bands (local clock). BE 19:00-05:59 and PT
# 20:45-06:00 approximate the published Low windows; the DE band is the
# InfraGO Nacht window; IT's is assumed (see the IT SV note). There is no
# single European night band — a global constant here would misprice BE
# by hours.
NIGHT_BANDS = {
    "DE": ("23:00", "06:00"),
    "IT": ("22:00", "06:00"),
    "BE": ("19:00", "05:59"),
    "PT": ("20:45", "06:00"),
}

# Commuter peak bands for the peak/congestion surcharges (AT congestion
# surcharge, CH peak multiplier): Mon-Fri 06:00-09:00 and 16:00-19:00.
# Deliberately ACTIVE conservative defaults — a night train's approach
# into Wien Hbf or Zürich HB during the morning peak plausibly touches a
# declared high-load section; treating every unconfirmed approach as free
# would systematically understate cost in exactly the pattern night
# trains run. Weekday-only windows are priced downstream at 5/7
# (calc_tac.WEEKDAY_BLEND) — the model has clock minutes, not dates.
PEAK_BANDS = {
    "AT": dict(band1=("06:00", "09:00"), band2=("16:00", "19:00"), weekdays_only=True),
    "CH": dict(band1=("06:00", "09:00"), band2=("16:00", "19:00"), weekdays_only=True),
}
NT_REF

## Per-country parameters

Only components a country actually levies are listed. Anything unlisted is
completed as `missing` — an absence recorded, never an implied zero. A
documented zero is `not_levied` and is a different claim.

In [ ]:
def S(cc, p, v, unit, src, loc, yr, ccy="EUR", note=""):
    return SV(cc, p, v, unit, SOURCED, src, loc, ccy, yr, note)


def D(cc, p, v, unit, formula, yr, src, loc, ccy="EUR"):
    """A derived value still names the document its inputs came from — otherwise
    provenance stops at the arithmetic and the seed row has no pointer."""
    return SV(cc, p, v, unit, DERIVED, src, loc, ccy, yr, formula)


TKM, GTKM = "EUR/train-km", "EUR/gross-tonne-km"
VALUES: list[SV] = []
MODE: dict[str, str] = {}

MODE["AT"] = "none"
VALUES += [
    S(
        "AT",
        "b_day",
        0.643,
        TKM,
        "AT-SNNB-2027",
        "Tab.11 §5.3.4",
        2027,
        note="Personenfernverkehr",
    ),
    S("AT", "gamma", 0.002282, GTKM, "AT-SNNB-2027", "Tab.13 §5.3.4", 2027),
    S(
        "AT",
        "congestion_surcharge_eur_km",
        1.6081,
        TKM,
        "AT-SNNB-2027",
        "§5.4 überlastete Schienenwege",
        2027,
        note="flat surcharge on declared overloaded sections; blanket "
        "application to peak-overlapping run shares is the model's "
        "conservative assumption (PEAK_BANDS)",
    ),
]

MODE["BE"] = "time_band"
VALUES += [
    D(
        "BE",
        "b_night",
        2.720182,
        TKM,
        "direct cost 2.142772 + off-peak high-density mark-up 0.577410; density class assumed High",
        2026,
        "BE-NS-2027",
        "App F.2 sheets 2.1.1 / 2.1.2.3",
    )
]

MODE["BG"] = "none"
VALUES += [
    S("BG", "b_day", 0.2495, TKM, "BG-NRIC-2026", "§I", 2026),
    S("BG", "gamma", 0.00083, GTKM, "BG-NRIC-2026", "§I", 2026),
]

# CH has NO day/night TAC split — its 22:00-06:00 band belongs to
# ELECTRICITY pricing (see 03), not to track access. What CH does have in
# the peak: the NZV doubles the base path price on declared high-load
# sections (peak_multiplier below, bands in PEAK_BANDS).
MODE["CH"] = "none"
VALUES += [
    S(
        "CH",
        "b_day",
        2.50,
        "CHF/train-path-km",
        "CH-NZV-BAV",
        "Art.1 Anhang 1",
        2026,
        "CHF",
        "line category A; quality factor 1.0 for treaty cross-border paths",
    ),
    S(
        "CH",
        "peak_multiplier",
        2.0,
        "factor",
        "CH-NZV-BAV",
        "Art.1 Anhang 1",
        2026,
        note="doubles the day-rate term on declared high-load sections during "
        "the commuter peak; blanket application to peak-overlapping run "
        "shares is the model's conservative assumption (PEAK_BANDS)",
    ),
    S(
        "CH",
        "gamma",
        0.0036,
        "CHF/gross-tonne-km",
        "CH-NZV-BAV",
        "Art.1(3)(b)",
        2026,
        "CHF",
        "Basispreis Verschleiss proxy for the per-vehicle formula",
    ),
    S(
        "CH",
        "per_stop",
        2.0,
        "CHF/stop",
        "CH-NZV",
        "Art.19a(4)",
        2026,
        "CHF",
        "capacity element of the path price, not a station charge — belongs here",
    ),
    SV(
        "CH",
        "revenue_share",
        None,
        "",
        MISSING,
        note="authority-set Deckungsbeitrag percentage not published; scenario parameter",
    ),
]

MODE["CZ"] = "none"
VALUES += [
    S(
        "CZ",
        "gamma",
        0.08163,
        "CZK/gross-tonne-km",
        "CZ-NS-2027",
        "charging annex",
        2027,
        "CZK",
        "Px=1.0 passenger; kETCS=1.0, no discount claimed",
    )
]

# DE prices per time band (InfraGO Nacht 23:00-06:00, NIGHT_BANDS), with
# the SPFV widening rule: a train carrying night accommodation (couchette/
# sleeper/capsule) is priced Nacht over its ENTIRE German run — modelled
# timetable-independent from the composition (night_full_if_accommodation
# in tac_night_mode.csv). An earlier 'segment' mode here was wrong: the
# tariff is a band tariff, the accommodation rule only widens the band.
MODE["DE"] = "time_band"
VALUES += [
    SV(
        "DE",
        "b_night",
        2.76,
        TKM,
        ASSUMED,
        "DE-INB-2026",
        "Anlage 5.3",
        "EUR",
        2026,
        "INB 2026 prints 3.33; the -17% BNetzA SPFV re-approval is applied but its decision "
        "document is not in the source set. Revert to 3.33 if it does not hold for Nacht.",
        low=2.76,
        high=3.33,
    )
]

MODE["DK"] = "none"
VALUES += [
    S(
        "DK",
        "b_day",
        5.80,
        "DKK/train-km",
        "DK-BEK-2024",
        "infrastrukturafgifter",
        2025,
        "DKK",
        "NS 2027 carries no numbers and refers to the executive order",
    )
]

MODE["EE"] = "none"
VALUES += [
    S("EE", "b_day", 0.74, TKM, "EE-TTJA-2026", "rates table", 2026),
    S(
        "EE",
        "gamma",
        0.00299,
        GTKM,
        "EE-TTJA-2026",
        "rates table",
        2026,
        note="international passenger mark-up is zero",
    ),
]

MODE["ES"] = "none"
VALUES += [
    D(
        "ES",
        "b_day",
        5.3181,
        TKM,
        "Mode A 1.6767 + Mode B 3.6414, line type A (ES-BOE-2024 Art.4); standard-gauge night "
        "trains from FR stay on the HS network",
        2023,
        "ES-BOE-2024",
        "Art.4",
    ),
    S(
        "ES",
        "seat_km",
        0.022014,
        "EUR/seat-km",
        "ES-BOE-2024",
        "Art.5",
        2023,
        note="Madrid-Barcelona-Frontera surcharge, 2.2014 EUR per 100 seat-km",
    ),
]

MODE["FI"] = "none"
VALUES += [
    S(
        "FI",
        "gamma",
        0.002054,
        GTKM,
        "FI-NS-2027",
        "Tab.2 §5.3",
        2027,
        note="electric-supply-equipment charge excluded as energy",
    )
]

MODE["FR"] = "none"
VALUES += [
    S("FR", "b_day", 0.657, TKM, "FR-DRR-2027-A52", "App 5.2.2 UIC 2-6", 2027),
    S(
        "FR",
        "gamma",
        0.005705,
        GTKM,
        "FR-DRR-2027-A52",
        "App 5.2.2 UIC 2-6",
        2027,
        note="published as 5.705 EUR per 1000 CGT-km; RM shows '-' for night trains",
    ),
]

MODE["GR"] = "none"
VALUES += [
    D(
        "GR",
        "b_day",
        1.897,
        TKM,
        "2.64 x 1.1975 inflation x 0.60 recovery",
        2019,
        "GR-OSE-2026",
        "ch.6",
    ),
    D(
        "GR",
        "gamma",
        0.004024,
        GTKM,
        "0.00560 x 1.1975 x 0.60",
        2019,
        "GR-OSE-2026",
        "ch.6",
    ),
]

MODE["HR"] = "none"
VALUES += [
    D(
        "HR",
        "b_day",
        2.154,
        TKM,
        "T 2.10 (EuroNight) x L1 1.90 x Cvlkm 0.54; L1 assumed for international mainlines",
        2027,
        "HR-NS-2027",
        "§5.3",
    )
]

MODE["HU"] = "none"
VALUES += [
    S(
        "HU",
        "b_day",
        1143.0,
        "HUF/train-km",
        "HU-NS-2627",
        "Annex 5.2-6",
        2027,
        "HUF",
        "path ensuring 9 + track category I passenger rate 1134",
    ),
    S(
        "HU",
        "gamma",
        1.05,
        "HUF/gross-tonne-km",
        "HU-NS-2627",
        "Annex 5.2-6",
        2027,
        "HUF",
    ),
]

MODE["IE"] = "none"
VALUES += [
    S(
        "IE",
        "b_day",
        1.9268,
        TKM,
        "IE-NS-2027",
        "ch.6.2/6.3",
        2027,
        note="fixed track access charge is franchise-only",
    )
]

MODE["IT"] = "time_band"
VALUES += [
    S("IT", "b_day", 0.185, TKM, "IT-LISTINO", "Component A flat term", 2027),
    S(
        "IT",
        "b_night",
        2.31,
        TKM,
        "IT-LISTINO",
        "Component B Basic FOND Standard Notturno",
        2027,
        note="night band assumed 22:00-06:00, not confirmed in the Listino",
    ),
    S(
        "IT",
        "gamma",
        0.002707,
        GTKM,
        "IT-LISTINO",
        "Component A speed class 150-175",
        2027,
        note="speed class taken from route average speed at runtime",
    ),
]

MODE["LT"] = "none"
VALUES += [
    S("LT", "gamma", 0.0012, GTKM, "LT-LTG-2627", "tariff decision §5.3.2", 2027)
]

MODE["LU"] = "none"
VALUES += [
    D(
        "LU",
        "b_day",
        3.333,
        TKM,
        "cC 2.771 x alpha 1.1615 (>8 bodies) x beta 1.0355",
        2026,
        "LU-NS-2027",
        "§5.3.2",
    ),
    S(
        "LU",
        "fixed_per_train_km",
        0.05,
        "EUR/train-km",
        "LU-NS-2027",
        "§5.3.2",
        2026,
        note="path administration on a regular timetable path",
    ),
]

MODE["LV"] = "none"
VALUES += [
    S(
        "LV",
        "b_day",
        1.30,
        TKM,
        "LV-NS-2027",
        "§5.2",
        2026,
        note="international passenger within the EEA",
    ),
    S("LV", "gamma", 0.00106848, GTKM, "LV-NS-2027", "§5.2", 2026),
]

MODE["NL"] = "none"
VALUES += [
    S(
        "NL",
        "b_day",
        2.2252,
        TKM,
        "NL-NS-2027",
        "train path service, class 601-3200 t",
        2027,
        note="weight class resolved from composition at runtime; all mark-ups are zero",
    )
]

MODE["NO"] = "none"
VALUES += [
    S(
        "NO",
        "b_day",
        9.94,
        "NOK/train-km",
        "NO-NS-2027",
        "Tab.4 §5.3.3",
        2026,
        "NOK",
        "non-Oslo rate applied uniformly, conservative",
    )
]

MODE["PL"] = "none"
VALUES += [
    S(
        "PL",
        "b_day",
        8.01,
        "PLN/train-km",
        "PL-PLK-A91",
        "Annex 9.1 SMK",
        2027,
        "PLN",
        "x WM(mass) x WK(line category), both 1.0 for NT-REF",
    )
]

MODE["PT"] = "time_band"
VALUES += [
    S("PT", "b_night", 2.16, TKM, "PT-IP-2027", "§5.3 cat A Low electric", 2027),
    S("PT", "b_day", 2.55, TKM, "PT-IP-2027", "§5.3 cat A Regular/Peak electric", 2027),
]

MODE["RO"] = "none"
VALUES += [
    D(
        "RO",
        "b_day",
        13.22,
        "RON/train-km",
        "Tc 9.562 + Ttsn 3.45 x [1 + (m-60) x 0.00014] at 600 t, class A; reproduces the worked "
        "example in Annex 26.a exactly",
        2024,
        "RO-CFR-A25",
        "Annex 25.a",
        "RON",
    )
]

MODE["SE"] = "none"
VALUES += [
    S("SE", "b_day", 4.98, "SEK/train-km", "SE-NS-2027", "Annex 1B", 2027, "SEK"),
    S(
        "SE",
        "gamma",
        0.0218,
        "SEK/gross-tonne-km",
        "SE-NS-2027",
        "Annex 1B",
        2027,
        "SEK",
        "passenger, mean axle load <=17 t",
    ),
]

MODE["SI"] = "none"
VALUES += [
    D(
        "SI",
        "b_day",
        2.41,
        TKM,
        "C_P1 2.01 x PP 1.44 (R4) x PD 1.05 x PM 0.75 x PV 0.97 x Pl 1.09; ETCS incentive not "
        "claimed",
        2027,
        "SI-NS-2027",
        "§5.3",
    )
]

MODE["SK"] = "none"
VALUES += [
    D(
        "SK",
        "b_day",
        1.0661,
        TKM,
        "U1 0.0691 + U2 0.997, track category 1",
        2019,
        "SK-ZSR-A52B",
        "Measure 2/2018 Annex 1",
    ),
    S(
        "SK",
        "gamma",
        0.001102,
        GTKM,
        "SK-ZSR-A52B",
        "Measure 2/2018 Annex 1 U3",
        2019,
        note="published as 1.102 EUR per 1000 gtkm",
    ),
]

MODE["UK"] = "none"
VALUES += [
    D(
        "UK",
        "b_day",
        2.2470,
        "GBP/train-km",
        "(loco 127.05 + 10 x coach 23.45) pence per vehicle-mile / 1.609344",
        2024,
        "UK-NR-CP7",
        "Default Passenger VUC",
        "GBP",
    )
]

for cc in ("CY", "MT"):
    MODE[cc] = "none"
    VALUES += [SV(cc, "b_day", None, "", NO_RAILWAY, note="no railway network")]

print(f"{len(MODE)} countries, {len(VALUES)} values before completion")

## Passage charges

Fixed per-train charges tied to a crossing rather than a country. Kept as their
own table because they are triggered by route geometry, and the Channel Tunnel
additionally carries a per-passenger term that couples to the demand model.

In [ ]:
PASSAGE_FIELDS = [
    "crossing_id",
    "charged_by",
    "parameter",
    "value",
    "unit",
    "currency",
    "basis_year",
    "source_id",
    "locator",
    "note",
]
PASSAGE = [
    dict(
        crossing_id="STOREBAELT",
        charged_by="Banedanmark",
        parameter="fixed_per_train",
        value=4876.73,
        unit="DKK/train",
        currency="DKK",
        basis_year=2025,
        source_id="DK-BEK-2024",
        locator="infrastrukturafgifter",
        note="excl. VAT",
    ),
    dict(
        crossing_id="OERESUND_DK",
        charged_by="Banedanmark",
        parameter="fixed_per_train",
        value=2592.02,
        unit="DKK/train",
        currency="DKK",
        basis_year=2025,
        source_id="DK-BEK-2024",
        locator="infrastrukturafgifter",
        note="Danish part",
    ),
    dict(
        crossing_id="OERESUND_SE",
        charged_by="Trafikverket",
        parameter="fixed_per_train",
        value=0.0,
        unit="SEK/train",
        currency="SEK",
        basis_year=2027,
        source_id="SE-NS-2027",
        locator="Annex 1B",
        note="passage charge applies to freight only; regular track and path charges still apply",
    ),
    dict(
        crossing_id="CHANNEL_TUNNEL",
        charged_by="Getlink",
        parameter="fixed_per_train",
        value=4039.0,
        unit="EUR/train one-way",
        currency="EUR",
        basis_year=2020,
        source_id="CT-GETLINK-2026",
        locator="Annexe 4 Offer 1",
        note="night trains run at 120 km/h off-peak by definition; 2020 basis, RPI/IPC indexed",
    ),
    dict(
        crossing_id="CHANNEL_TUNNEL",
        charged_by="Getlink",
        parameter="per_passenger",
        value=18.35,
        unit="EUR/passenger one-way",
        currency="EUR",
        basis_year=2020,
        source_id="CT-GETLINK-2026",
        locator="Annexe 4 Offer 1",
        note="couples the crossing cost to the demand model, not to routing",
    ),
]
write_data("passage_charges.csv", PASSAGE_FIELDS, PASSAGE)

# Passage geometry (data/passage_geometries.geojson, from QGIS — WGS84
# crossing polygons) is the routing-side half of the passage model: a trip
# leg intersecting a polygon owns the crossing. OERESUND_DK/_SE share the
# single OERESUND polygon — each IM bills its half of one crossing.
import json as _json

with open(DATA_DIR / "passage_geometries.geojson", encoding="utf-8") as fh:
    _geo = _json.load(fh)
GEOMETRY_BY_CROSSING = {
    f["properties"]["crossing_id"]: f["geometry"] for f in _geo["features"]
}
_geom_key = lambda cid: "OERESUND" if cid.startswith("OERESUND") else cid
for _row in PASSAGE:
    assert _geom_key(_row["crossing_id"]) in GEOMETRY_BY_CROSSING, (
        f"no geometry for {_row['crossing_id']}"
    )
print(f"passage geometry coverage OK ({len(GEOMETRY_BY_CROSSING)} polygons)")

## Complete the grid and emit

In [ ]:
PARAMS = (
    "b_day",
    "b_night",
    "gamma",
    "per_stop",
    "seat_km",
    "revenue_share",
    "fixed_per_train_km",
    "peak_multiplier",
    "congestion_surcharge_eur_km",
)
have = {(v.country_code, v.parameter) for v in VALUES}
for cc in MODE:
    no_rail = any(v.country_code == cc and v.status == NO_RAILWAY for v in VALUES)
    for p in PARAMS:
        if (cc, p) in have:
            continue
        VALUES.append(
            SV(
                cc,
                p,
                None,
                "",
                NO_RAILWAY if no_rail else MISSING,
                note="no railway network" if no_rail else "component not documented",
            )
        )

# A country with no usable charging term at all is a calibration failure, not a
# tariff fact — catch it here rather than in the seed.
broken = [
    cc
    for cc in MODE
    if not any(
        v.country_code == cc
        and v.parameter in ("b_day", "b_night", "gamma")
        and v.status in USABLE
        for v in VALUES
    )
    and cc not in ("CY", "MT")
]
assert not broken, f"countries with no usable charging term: {broken}"

write_data(
    "tac_night_mode.csv",
    [
        "country_code",
        "night_mode",
        "night_band_start",
        "night_band_end",
        "night_full_if_accommodation",
    ],
    [
        dict(
            country_code=cc,
            night_mode=m,
            night_band_start=NIGHT_BANDS[cc][0] if m == "time_band" else "",
            night_band_end=NIGHT_BANDS[cc][1] if m == "time_band" else "",
            night_full_if_accommodation=(cc == "DE"),
        )  # InfraGO SPFV widening
        for cc, m in sorted(MODE.items())
    ],
)

write_data(
    "tac_peak_bands.csv",
    [
        "country_code",
        "band1_start",
        "band1_end",
        "band2_start",
        "band2_end",
        "weekdays_only",
    ],
    [
        dict(
            country_code=cc,
            band1_start=b["band1"][0],
            band1_end=b["band1"][1],
            band2_start=b["band2"][0],
            band2_end=b["band2"][1],
            weekdays_only=b["weekdays_only"],
        )
        for cc, b in sorted(PEAK_BANDS.items())
    ],
)
emit("tac_components.csv", sorted(VALUES, key=lambda v: (v.country_code, v.parameter)))

In [ ]:
# Display / validation only — pandas is fine here, seed.py skips this cell.
import pandas as pd

_df = pd.DataFrame([asdict(v) for v in VALUES])
_reg = set(pd.read_csv(DATA_DIR / "sources_register.csv")["source_id"])
_unknown = set(_df.loc[_df.source_id.ne(""), "source_id"]) - _reg
assert not _unknown, f"unregistered source ids: {sorted(_unknown)}"
_bad = _df[(_df.status == "assumed") & (_df.low.isna() | _df.high.isna())]
assert _bad.empty, _bad
print(f"{len(_df)} values, {_df.source_id.ne('').sum()} with a source pointer")
_df.status.value_counts().to_frame("n")